In [2]:
import geopandas as gdp
import pandas as pd
from shapely import wkt
from pyarrow import Table
from shapely.geometry import Point

In [3]:
# Assuming geometry is in WKT format and stored in a column named 'geometry'
def wkt_to_wkb(table):
    wkb_geometry = [wkt.loads(geom).wkb if geom is not None else None for geom in table['geometry'].to_pylist()]
    return table.replace_column(table.schema.get_field_index('geometry'), Table.from_arrays([wkb_geometry], names=['geometry']))

In [4]:
df = pd.read_parquet(
    "s3://weave.energy/smart-meter.parquet", 
    # The 7pm evening peak on the first day all DNOs have data for
    filters=[("data_collection_log_timestamp", "==", pd.Timestamp("2024-2-12 19:00Z"))])

In [5]:
gdf = gdp.GeoDataFrame(
                df,
                geometry=[Point(geo_dict['x'], geo_dict['y']) for geo_dict in df['geometry']],
                crs="EPSG:4326"
)

In [6]:
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 104502 entries, 0 to 104501
Data columns (total 9 columns):
 #   Column                           Non-Null Count   Dtype              
---  ------                           --------------   -----              
 0   dataset_id                       104502 non-null  str                
 1   dno_alias                        104502 non-null  str                
 2   aggregated_device_count_active   104502 non-null  int64              
 3   total_consumption_active_import  104502 non-null  int64              
 4   data_collection_log_timestamp    104502 non-null  datetime64[ms, UTC]
 5   geometry                         104502 non-null  geometry           
 6   secondary_substation_unique_id   104502 non-null  str                
 7   lv_feeder_unique_id              104502 non-null  str                
 8   bbox                             104502 non-null  object             
dtypes: datetime64[ms, UTC](1), geometry(1), int64(2), ob

In [7]:
gdf.head(20)

,dataset_id,dno_alias,aggregated_device_count_active,total_consumption_active_import,data_collection_log_timestamp,geometry,secondary_substation_unique_id,lv_feeder_unique_id,bbox
0,NGED_110283_11_Feb_2024,NGED,19,9225,2024-02-12 19:00:00+00:00,POINT (-2.6402 51.4845),110283-Newlyn Ave P,11-11,"{'xmin': -2.6402, 'ymin': 51.4845, 'xmax': -2...."
1,NGED_110283_21_Feb_2024,NGED,14,2583,2024-02-12 19:00:00+00:00,POINT (-2.6402 51.4845),110283-Newlyn Ave P,21-21,"{'xmin': -2.6402, 'ymin': 51.4845, 'xmax': -2...."
2,NGED_110283_31_Feb_2024,NGED,8,2442,2024-02-12 19:00:00+00:00,POINT (-2.6402 51.4845),110283-Newlyn Ave P,31-31,"{'xmin': -2.6402, 'ymin': 51.4845, 'xmax': -2...."
3,NGED_110362_11_Feb_2024,NGED,10,5095,2024-02-12 19:00:00+00:00,POINT (-2.5521 51.426),110362-Westbrook Road Id,11-11,"{'xmin': -2.5521, 'ymin': 51.426, 'xmax': -2.5..."
4,NGED_110362_21_Feb_2024,NGED,16,4629,2024-02-12 19:00:00+00:00,POINT (-2.5521 51.426),110362-Westbrook Road Id,21-21,"{'xmin': -2.5521, 'ymin': 51.426, 'xmax': -2.5..."
5,NGED_110362_31_Feb_2024,NGED,27,11387,2024-02-12 19:00:00+00:00,POINT (-2.5521 51.426),110362-Westbrook Road Id,31-31,"{'xmin': -2.5521, 'ymin': 51.426, 'xmax': -2.5..."
6,NGED_110670_11_Feb_2024,NGED,27,6613,2024-02-12 19:00:00+00:00,POINT (-2.5544 51.4461),110670-Hardenhuish Rd Id,11-11,"{'xmin': -2.5544, 'ymin': 51.4461, 'xmax': -2...."
7,NGED_110670_21_Feb_2024,NGED,34,18185,2024-02-12 19:00:00+00:00,POINT (-2.5544 51.4461),110670-Hardenhuish Rd Id,21-21,"{'xmin': -2.5544, 'ymin': 51.4461, 'xmax': -2...."
8,NGED_110670_31_Feb_2024,NGED,28,7337,2024-02-12 19:00:00+00:00,POINT (-2.5544 51.4461),110670-Hardenhuish Rd Id,31-31,"{'xmin': -2.5544, 'ymin': 51.4461, 'xmax': -2...."
9,NGED_110787_11_Feb_2024,NGED,14,4991,2024-02-12 19:00:00+00:00,POINT (-2.5625 51.5559),110787-Florence Park Od,11-11,"{'xmin': -2.5625, 'ymin': 51.5559, 'xmax': -2...."


In [8]:
gdf['composite_feeder_id'] = gdf['secondary_substation_unique_id'] + '-' + gdf['lv_feeder_unique_id']

## Column Explanations

1. dataset_id: Likely a unique identifier for each data collection batch or dataset
2. dno_alias: Distribution Network Operator alias - the company responsible for operating the  electricity distribution network in a specific region (e.g., UK Power Networks, Western Power Distribution)
3. aggregated_device_count_active: Number of active smart meters/devices being measured in this group
4. total_consumption_active_import: Total electricity consumption (import from grid) for this group of meters, likely in Wh (Watt-hours) or kWh
5. data_collection_log_timestamp: The timestamp when this data was collected (in UTC timezone)
6. geometry: Geographic point location of the substation representing where this measurement was taken
7. secondary_substation_unique_id: Identifier for the local distribution transformer/substation that converts medium voltage to low voltage for final distribution
8. lv_feeder_unique_id: Low Voltage feeder identifier - the specific power line that connects the secondary substation to end consumers
9. bbox: Likely "bounding box" - unfortunately the same geomtry as is found in the geometry column

In [0]:
print(f"Number of unique substations = {gdf['secondary_substation_unique_id'].nunique()}")
print(f"Number of unique LV Feeders = {gdf['lv_feeder_unique_id'].nunique()}")
print(f"Number of unique geometries = {gdf['geometry'].nunique()}")
print(f"Number of unique substations + LV feeders combinations = {(gdf['secondary_substation_unique_id'] + '-' + gdf['lv_feeder_unique_id']).nunique()}")
print(f"Number of unique substations + geometry combinations = {(gdf['secondary_substation_unique_id'] + '-' + str(gdf['geometry'])).nunique()}")
print(f"Number of unique LV Feeders + geometry combinations = {(gdf['lv_feeder_unique_id'] + '-' + str(gdf['geometry'])).nunique()}")

We can see below that every substation and geometry pair has a few LV feeders attached to it. Therefore we can see that the Lv feeder is lower level than the substation. Unfortunatley the LV feeder id is not actually globally unique. 

In [0]:
gdf.groupby(['secondary_substation_unique_id', 'geometry'])['lv_feeder_unique_id'].nunique()